# 파일 정보

- 설명: LightGBM 특징 조합, Optuna 튜닝, 임계값과 최종 성능을 탐색한 모델링 기록입니다.
- 작성자: 김동혁

### 상관계수 조합 구성

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report, roc_auc_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 설정
plt.rc('font', family='Malgun Gothic') 
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
print("\n[ 🚀 LightGBM을 활용한 블루리본 예측 최종 모델링 (과적합 방지 Optuna 튜닝) ]")

# =========================================================================
# 1. 데이터 불러오기 및 X, y 세팅
# =========================================================================
df = pd.read_csv('../data/블루리본_최최종_마참내_v1.2.csv', encoding='cp949')
y = df['블루리본 여부']

# 이미지 모델 결과 컬럼들의 0 값을 NaN으로 변경 (LightGBM 특화 처리)
image_cols = [
    '고급성_평균', '고급성_중앙값', 
    '쾌적성_평균', '쾌적성_중앙값', 
    '감성_평균', '감성_중앙값'
]

df[image_cols] = df[image_cols].replace(0, np.nan)
print("ℹ️ 이미지 변수 내 0 -> NaN 변환 완료!")

# ml_lgbm.py 기준 적용 (가중치 점수는 살리고, 순수 score 컬럼은 제외)
# drop_cols = ['카테고리', '매장명', '블루리본 여부', 'idx_family_weighted_score',
#              '고급성_중앙값', '쾌적성_중앙값', '감성_중앙값']
drop_cols = ['카테고리', '매장명', '블루리본 여부', 'idx_family_weighted_score',
             '고급성_평균', '쾌적성_평균', '감성_평균']
X_base = df.drop(columns=drop_cols)

print(f"✅ 사용된 총 변수 개수: {len(X_base.columns)}개")

In [ ]:
# =========================================================================
# 2. Train / Test 분리 및 전처리
# =========================================================================
# random_state, test_size, stratify 고정 (기존 파일과 완벽히 동일한 데이터 분할)
X_train, X_test, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# =========================================================================
# 3. 상관계수 높은 변수쌍 기준 32개 제거 조합 실험
# =========================================================================

import itertools
from sklearn.model_selection import StratifiedKFold

print("\n[ 🔁 상관계수 높은 변수쌍 기준 32개 조합 실험 시작 ]")

# -------------------------------------------------------------------------
# 3-1. 상관계수 높은 변수쌍 정의
# 각 행에서 왼쪽 변수 또는 오른쪽 변수 중 하나만 제거
# -------------------------------------------------------------------------

corr_pairs = [
    ("idx_anniversary_weighted_score", "idx_luxury_weighted_score"),
    ("idx_date_weighted_score", "idx_ambiance_weighted_score"),
    # ("고급성_평균", "감성_평균"),
    ("고급성_중앙값", "감성_중앙값"),
    ("idx_service_weighted_score", "idx_luxury_weighted_score")
]

print("\n제거 후보 변수쌍:")
for i, (v1, v2) in enumerate(corr_pairs, start=1):
    print(f"{i}. {v1}  vs  {v2}")

# -------------------------------------------------------------------------
# 3-2. CV 설정
# -------------------------------------------------------------------------

N_TRIALS = 50      # 오래 걸리면 20으로 줄이면 됨
CV_SPLITS = 5

cv = StratifiedKFold(
    n_splits=CV_SPLITS,
    shuffle=True,
    random_state=42
)

# -------------------------------------------------------------------------
# 3-3. 전체 조합 생성
# 0이면 변수1 제거, 1이면 변수2 제거
# -------------------------------------------------------------------------

all_combinations = list(itertools.product([0, 1], repeat=len(corr_pairs)))

print(f"\n✅ 총 실험 조합 개수: {len(all_combinations)}개")
print("각 조합마다 상관쌍 5개에서 하나씩 선택하여 제거합니다.")

# 결과 저장용
results = []

# 최적 모델 저장용
best_models = {}

In [ ]:
# =========================================================================
# 4. 32개 조합 반복 실행
# =========================================================================

for combo_idx, combo in enumerate(all_combinations, start=1):
    
    print("\n" + "=" * 90)
    print(f"[진행상황] {combo_idx} / {len(all_combinations)} 번째 조합 실행 중")
    print("=" * 90)
    
    # ---------------------------------------------------------------------
    # 4-1. 이번 조합에서 제거할 변수 선택
    # ---------------------------------------------------------------------
    
    selected_drop_cols = []
    
    for choice, (var1, var2) in zip(combo, corr_pairs):
        if choice == 0:
            selected_drop_cols.append(var1)
        else:
            selected_drop_cols.append(var2)
    
    # 중복 제거
    # idx_luxury_weighted_score가 두 번 등장하므로 실제 제거 개수는 4개가 될 수도 있음
    selected_drop_cols = list(dict.fromkeys(selected_drop_cols))
    
    # 실제 X_train에 존재하는 컬럼만 제거
    existing_drop_cols = [col for col in selected_drop_cols if col in X_train.columns]
    missing_drop_cols = [col for col in selected_drop_cols if col not in X_train.columns]
    
    print("\n이번 조합에서 제거할 변수:")
    for col in existing_drop_cols:
        print(f" - {col}")
    
    if len(missing_drop_cols) > 0:
        print("\n⚠️ 데이터에 존재하지 않아 제거하지 못한 변수:")
        for col in missing_drop_cols:
            print(f" - {col}")
    
    # ---------------------------------------------------------------------
    # 4-2. 변수 제거한 데이터 생성
    # ---------------------------------------------------------------------
    
    X_train_sub = X_train.drop(columns=existing_drop_cols)
    X_test_sub = X_test.drop(columns=existing_drop_cols)
    
    print(f"\n사용 변수 개수: {X_train_sub.shape[1]}개")
    print(f"실제 제거 변수 개수: {len(existing_drop_cols)}개")
    
    # ---------------------------------------------------------------------
    # 4-3. Optuna 목적 함수 정의
    # ---------------------------------------------------------------------
    
    def objective_lgbm_regularized(trial):
        params = {
            # 기본 설정
            "objective": "binary",
            "metric": "binary_logloss",
            "boosting_type": "gbdt",
            "random_state": 42,
            "verbose": -1,
            "n_jobs": -1,

            # 모델 복잡도 제어
            "n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 7, 31),
            "max_depth": trial.suggest_int("max_depth", 2, 6),

            # 과적합 방지용 샘플링
            "subsample": trial.suggest_float("subsample", 0.5, 0.85),
            "subsample_freq": trial.suggest_int("subsample_freq", 1, 5),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.85),

            # 리프 최소 데이터 수
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 50),
            "min_child_weight": trial.suggest_float("min_child_weight", 0.001, 10.0, log=True),

            # 규제
            "reg_alpha": trial.suggest_float("reg_alpha", 0.01, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.01, 10.0, log=True),

            # 분할 이득 제한
            "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0)
        }
        
        model = lgb.LGBMClassifier(**params)
        
        scores = cross_val_score(
            model,
            X_train_sub,
            y_train,
            cv=cv,
            scoring="roc_auc",
            n_jobs=-1
        )
        
        return np.mean(scores)
    
    # ---------------------------------------------------------------------
    # 4-4. Optuna 튜닝 실행
    # ---------------------------------------------------------------------
    
    study = optuna.create_study(
        direction="maximize",
        study_name=f"LGBM_combo_{combo_idx}"
    )
    
    study.optimize(
        objective_lgbm_regularized,
        n_trials=N_TRIALS,
        show_progress_bar=False
    )
    
    best_params = study.best_params
    
    print("\n--- Optuna 튜닝 완료 ---")
    print(f"Best CV ROC-AUC: {study.best_value:.4f}")
    print("Best Params:")
    print(best_params)
    
    # ---------------------------------------------------------------------
    # 4-5. 최적 파라미터로 최종 모델 학습
    # ---------------------------------------------------------------------
    
    best_lgbm_model = lgb.LGBMClassifier(
        **best_params,
        objective="binary",
        metric="binary_logloss",
        boosting_type="gbdt",
        random_state=42,
        verbose=-1,
        n_jobs=-1
    )
    
    best_lgbm_model.fit(X_train_sub, y_train)
    
    # ---------------------------------------------------------------------
    # 4-6. Train / Test 예측
    # ---------------------------------------------------------------------
    
    y_train_pred = best_lgbm_model.predict(X_train_sub)
    y_test_pred = best_lgbm_model.predict(X_test_sub)
    
    y_train_proba = best_lgbm_model.predict_proba(X_train_sub)[:, 1]
    y_test_proba = best_lgbm_model.predict_proba(X_test_sub)[:, 1]
    
    # ---------------------------------------------------------------------
    # 4-7. 성능 평가
    # ---------------------------------------------------------------------
    
    train_auc = roc_auc_score(y_train, y_train_proba)
    test_auc = roc_auc_score(y_test, y_test_proba)
    auc_gap = abs(train_auc - test_auc)
    
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    
    train_f1 = f1_score(y_train, y_train_pred)
    test_f1 = f1_score(y_test, y_test_pred)
    
    test_precision = precision_score(y_test, y_test_pred)
    test_recall = recall_score(y_test, y_test_pred)
    
    print("\n[Train vs Test 과적합 검증]")
    print(f"Train ROC-AUC : {train_auc:.4f}")
    print(f"Test ROC-AUC  : {test_auc:.4f}")
    print(f"AUC Gap       : {auc_gap:.4f}")
    
    print("\n[Test 성능]")
    print(f"Accuracy  : {test_acc:.4f}")
    print(f"Precision : {test_precision:.4f}")
    print(f"Recall    : {test_recall:.4f}")
    print(f"F1-score  : {test_f1:.4f}")
    
    # ---------------------------------------------------------------------
    # 4-8. 결과 저장
    # ---------------------------------------------------------------------
    
    result_row = {
        "조합번호": combo_idx,
        "선택패턴": combo,
        "제거변수": ", ".join(existing_drop_cols),
        "제거변수개수": len(existing_drop_cols),
        "사용변수개수": X_train_sub.shape[1],
        
        "CV_ROC_AUC": study.best_value,
        
        "Train_ROC_AUC": train_auc,
        "Test_ROC_AUC": test_auc,
        "AUC_Gap": auc_gap,
        
        "Train_Accuracy": train_acc,
        "Test_Accuracy": test_acc,
        
        "Train_F1": train_f1,
        "Test_F1": test_f1,
        
        "Test_Precision": test_precision,
        "Test_Recall": test_recall,
        
        "Best_Params": best_params
    }
    
    results.append(result_row)
    
    best_models[combo_idx] = {
        "model": best_lgbm_model,
        "drop_cols": existing_drop_cols,
        "features": X_train_sub.columns.tolist(),
        "study": study,
        "best_params": best_params
    }



In [ ]:
# =========================================================================
# 5. 전체 결과표 생성 및 저장
# =========================================================================

results_df = pd.DataFrame(results)

results_df_sorted = results_df.sort_values(
    by=["Test_ROC_AUC", "AUC_Gap", "Test_F1"],
    ascending=[False, True, False]
).reset_index(drop=True)

print("\n" + "=" * 90)
print("🏆 LightGBM 32개 조합 전체 결과 Top 10")
print("=" * 90)

display_cols = [
    "조합번호",
    "제거변수",
    "제거변수개수",
    "사용변수개수",
    "CV_ROC_AUC",
    "Train_ROC_AUC",
    "Test_ROC_AUC",
    "AUC_Gap",
    "Test_Accuracy",
    "Test_F1",
    "Test_Precision",
    "Test_Recall"
]

print(results_df_sorted[display_cols].head(10))

# CSV 저장
results_df_sorted.to_csv(
    "../data/lgbm_상관변수_32조합_제거실험결과.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ 결과 저장 완료: ../data/lgbm_상관변수_32조합_제거실험결과.csv")

### Threshold 추가하여 정확도 높이려는 코드

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report, roc_auc_score, f1_score, precision_recall_curve
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 설정
plt.rc('font', family='Malgun Gothic') 
plt.rcParams['axes.unicode_minus'] = False


print("\n[ 🚀 LightGBM을 활용한 블루리본 예측 최종 모델링 (과적합 방지 Optuna 튜닝) ]")

# =========================================================================
# 1. 데이터 불러오기 및 X, y 세팅
# =========================================================================
df = pd.read_csv('../data/블루리본_최최종_마참내_v1.2.csv', encoding='cp949')
y = df['블루리본 여부']

# 이미지 모델 결과 컬럼들의 0 값을 NaN으로 변경 (LightGBM 특화 처리)
image_cols = [
    '고급성_평균', '고급성_중앙값', 
    '쾌적성_평균', '쾌적성_중앙값', 
    '감성_평균', '감성_중앙값'
]

df[image_cols] = df[image_cols].replace(0, np.nan)
print("ℹ️ 이미지 변수 내 0 -> NaN 변환 완료!")

# ml_lgbm.py 기준 적용 (가중치 점수는 살리고, 순수 score 컬럼은 제외)
drop_cols = ['카테고리', '매장명', '블루리본 여부', 'idx_family_weighted_score',
             '고급성_중앙값', '쾌적성_중앙값', '감성_중앙값']
# drop_cols = ['카테고리', '매장명', '블루리본 여부', 'idx_family_weighted_score',
#              '고급성_평균', '쾌적성_평균', '감성_평균']
X_base = df.drop(columns=drop_cols)

print(f"✅ 사용된 총 변수 개수: {len(X_base.columns)}개")


# =========================================================================
# 2. Train / Test 분리 및 전처리
# =========================================================================
# random_state, test_size, stratify 고정 (기존 파일과 완벽히 동일한 데이터 분할)
X_train, X_test, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42, stratify=y
)


# =========================================================================
# 3. 상관계수 높은 변수쌍 기준 32개 제거 조합 실험
# =========================================================================

import itertools
from sklearn.model_selection import StratifiedKFold

print("\n[ 🔁 상관계수 높은 변수쌍 기준 32개 조합 실험 시작 ]")

# -------------------------------------------------------------------------
# 3-1. 상관계수 높은 변수쌍 정의
# 각 행에서 왼쪽 변수 또는 오른쪽 변수 중 하나만 제거
# -------------------------------------------------------------------------

corr_pairs = [
    ("idx_anniversary_weighted_score", "idx_luxury_weighted_score"),
    ("idx_date_weighted_score", "idx_ambiance_weighted_score"),
    ("고급성_평균", "감성_평균"),
    # ("고급성_중앙값", "감성_중앙값"),
    ("idx_service_weighted_score", "idx_luxury_weighted_score")
]

print("\n제거 후보 변수쌍:")
for i, (v1, v2) in enumerate(corr_pairs, start=1):
    print(f"{i}. {v1}  vs  {v2}")

# -------------------------------------------------------------------------
# 3-2. CV 설정
# -------------------------------------------------------------------------

N_TRIALS = 50      # 오래 걸리면 20으로 줄이면 됨
CV_SPLITS = 5

cv = StratifiedKFold(
    n_splits=CV_SPLITS,
    shuffle=True,
    random_state=42
)

# -------------------------------------------------------------------------
# 3-3. 전체 조합 생성
# 0이면 변수1 제거, 1이면 변수2 제거
# -------------------------------------------------------------------------

all_combinations = list(itertools.product([0, 1], repeat=len(corr_pairs)))

print(f"\n✅ 총 실험 조합 개수: {len(all_combinations)}개")
print("각 조합마다 상관쌍 5개에서 하나씩 선택하여 제거합니다.")

# 결과 저장용
results = []

# 최적 모델 저장용
best_models = {}


# =========================================================================
# 4. 32개 조합 반복 실행
# =========================================================================

for combo_idx, combo in enumerate(all_combinations, start=1):
    
    print("\n" + "=" * 90)
    print(f"[진행상황] {combo_idx} / {len(all_combinations)} 번째 조합 실행 중")
    print("=" * 90)
    
    # ---------------------------------------------------------------------
    # 4-1. 이번 조합에서 제거할 변수 선택
    # ---------------------------------------------------------------------
    
    selected_drop_cols = []
    
    for choice, (var1, var2) in zip(combo, corr_pairs):
        if choice == 0:
            selected_drop_cols.append(var1)
        else:
            selected_drop_cols.append(var2)
    
    # 중복 제거
    # idx_luxury_weighted_score가 두 번 등장하므로 실제 제거 개수는 4개가 될 수도 있음
    selected_drop_cols = list(dict.fromkeys(selected_drop_cols))
    
    # 실제 X_train에 존재하는 컬럼만 제거
    existing_drop_cols = [col for col in selected_drop_cols if col in X_train.columns]
    missing_drop_cols = [col for col in selected_drop_cols if col not in X_train.columns]
    
    print("\n이번 조합에서 제거할 변수:")
    for col in existing_drop_cols:
        print(f" - {col}")
    
    if len(missing_drop_cols) > 0:
        print("\n⚠️ 데이터에 존재하지 않아 제거하지 못한 변수:")
        for col in missing_drop_cols:
            print(f" - {col}")
    
    # ---------------------------------------------------------------------
    # 4-2. 변수 제거한 데이터 생성
    # ---------------------------------------------------------------------
    
    X_train_sub = X_train.drop(columns=existing_drop_cols)
    X_test_sub = X_test.drop(columns=existing_drop_cols)
    
    print(f"\n사용 변수 개수: {X_train_sub.shape[1]}개")
    print(f"실제 제거 변수 개수: {len(existing_drop_cols)}개")
    
    # ---------------------------------------------------------------------
    # 4-3. Optuna 목적 함수 정의
    # ---------------------------------------------------------------------
    
    def objective_lgbm_regularized(trial):
        params = {
            # 기본 설정
            "objective": "binary",
            "metric": "binary_logloss",
            "boosting_type": "gbdt",
            "random_state": 42,
            "verbose": -1,
            "n_jobs": -1,

            # 모델 복잡도 제어
            "n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 7, 31),
            "max_depth": trial.suggest_int("max_depth", 2, 6),

            # 과적합 방지용 샘플링
            "subsample": trial.suggest_float("subsample", 0.5, 0.85),
            "subsample_freq": trial.suggest_int("subsample_freq", 1, 5),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.85),

            # 리프 최소 데이터 수
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 50),
            "min_child_weight": trial.suggest_float("min_child_weight", 0.001, 10.0, log=True),

            # 규제
            "reg_alpha": trial.suggest_float("reg_alpha", 0.01, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.01, 10.0, log=True),

            # [추가] 데이터 불균형 해소를 위한 양성(블루리본) 클래스 가중치 탐색
            "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 5.0),

            # 분할 이득 제한
            "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0)
        }
        
        model = lgb.LGBMClassifier(**params)
        
        scores = cross_val_score(
            model,
            X_train_sub,
            y_train,
            cv=cv,
            scoring="roc_auc",
            n_jobs=-1
        )
        
        return np.mean(scores)
    
    # ---------------------------------------------------------------------
    # 4-4. Optuna 튜닝 실행
    # ---------------------------------------------------------------------
    
    study = optuna.create_study(
        direction="maximize",
        study_name=f"LGBM_combo_{combo_idx}"
    )
    
    study.optimize(
        objective_lgbm_regularized,
        n_trials=N_TRIALS,
        show_progress_bar=False
    )
    
    best_params = study.best_params
    
    print("\n--- Optuna 튜닝 완료 ---")
    print(f"Best CV ROC-AUC: {study.best_value:.4f}")
    print("Best Params:")
    print(best_params)
    
    # ---------------------------------------------------------------------
    # 4-5. 최적 파라미터로 최종 모델 학습
    # ---------------------------------------------------------------------
    
    best_lgbm_model = lgb.LGBMClassifier(
        **best_params,
        objective="binary",
        metric="binary_logloss",
        boosting_type="gbdt",
        random_state=42,
        verbose=-1,
        n_jobs=-1
    )
    
    best_lgbm_model.fit(X_train_sub, y_train)
    
    # ---------------------------------------------------------------------
    # 4-6. Train / Test 예측
    # ---------------------------------------------------------------------
    
    y_train_proba = best_lgbm_model.predict_proba(X_train_sub)[:, 1]
    y_test_proba = best_lgbm_model.predict_proba(X_test_sub)[:, 1]
    
    # [추가] 최적 임계값(Threshold) 찾기 (Train 데이터 기준 F1 Score 최대화)
    precisions, recalls, thresholds = precision_recall_curve(y_train, y_train_proba)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
    best_threshold = thresholds[np.argmax(f1_scores)]
    print(f"   💡 모델 최적 예측 임계값(Threshold): {best_threshold:.4f} (기본 0.5에서 변경)")
    
    # 최적 임계값을 적용하여 최종 예측값(0 또는 1) 산출
    y_train_pred = (y_train_proba >= best_threshold).astype(int)
    y_test_pred = (y_test_proba >= best_threshold).astype(int)

    # ---------------------------------------------------------------------
    # 4-7. 성능 평가
    # ---------------------------------------------------------------------
    
    train_auc = roc_auc_score(y_train, y_train_proba)
    test_auc = roc_auc_score(y_test, y_test_proba)
    auc_gap = abs(train_auc - test_auc)
    
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    
    train_f1 = f1_score(y_train, y_train_pred)
    test_f1 = f1_score(y_test, y_test_pred)
    
    test_precision = precision_score(y_test, y_test_pred)
    test_recall = recall_score(y_test, y_test_pred)
    
    print("\n[Train vs Test 과적합 검증]")
    print(f"Train ROC-AUC : {train_auc:.4f}")
    print(f"Test ROC-AUC  : {test_auc:.4f}")
    print(f"AUC Gap       : {auc_gap:.4f}")
    
    print("\n[Test 성능]")
    print(f"Accuracy  : {test_acc:.4f}")
    print(f"Precision : {test_precision:.4f}")
    print(f"Recall    : {test_recall:.4f}")
    print(f"F1-score  : {test_f1:.4f}")
    
    # ---------------------------------------------------------------------
    # 4-8. 결과 저장
    # ---------------------------------------------------------------------
    
    result_row = {
        "조합번호": combo_idx,
        "선택패턴": combo,
        "제거변수": ", ".join(existing_drop_cols),
        "제거변수개수": len(existing_drop_cols),
        "사용변수개수": X_train_sub.shape[1],
        
        "CV_ROC_AUC": study.best_value,
        
        "Train_ROC_AUC": train_auc,
        "Test_ROC_AUC": test_auc,
        "AUC_Gap": auc_gap,
        
        "Train_Accuracy": train_acc,
        "Test_Accuracy": test_acc,
        
        "Train_F1": train_f1,
        "Test_F1": test_f1,
        
        "Test_Precision": test_precision,
        "Test_Recall": test_recall,
        
        "Best_Params": best_params
    }
    
    results.append(result_row)
    
    best_models[combo_idx] = {
        "model": best_lgbm_model,
        "drop_cols": existing_drop_cols,
        "features": X_train_sub.columns.tolist(),
        "study": study,
        "best_params": best_params
    }


# =========================================================================
# 5. 전체 결과표 생성 및 저장
# =========================================================================

results_df = pd.DataFrame(results)

results_df_sorted = results_df.sort_values(
    by=["Test_ROC_AUC", "AUC_Gap", "Test_F1"],
    ascending=[False, True, False]
).reset_index(drop=True)

print("\n" + "=" * 90)
print("🏆 LightGBM 32개 조합 전체 결과 Top 10")
print("=" * 90)

display_cols = [
    "조합번호",
    "제거변수",
    "제거변수개수",
    "사용변수개수",
    "CV_ROC_AUC",
    "Train_ROC_AUC",
    "Test_ROC_AUC",
    "AUC_Gap",
    "Test_Accuracy",
    "Test_F1",
    "Test_Precision",
    "Test_Recall"
]

print(results_df_sorted[display_cols].head(10))

# CSV 저장
results_df_sorted.to_csv(
    "../data/lgbm_상관변수_32조합_제거실험결과.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ 결과 저장 완료: ../data/lgbm_상관변수_32조합_제거실험결과.csv")

### 앙상블 voting

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import optuna
import itertools

from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    classification_report, roc_auc_score, f1_score, precision_recall_curve
)

import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rc('font', family='Malgun Gothic') 
plt.rcParams['axes.unicode_minus'] = False

print("\n[ 🚀 LightGBM + XGBoost 앙상블 (소프트 보팅) 최종 모델링 ]")

# =========================================================================
# 1. 데이터 불러오기 및 X, y 세팅
# =========================================================================
df = pd.read_csv('../data/블루리본_최최종_마참내_v1.2.csv', encoding='cp949')
y = df['블루리본 여부']

image_cols = [
    '고급성_평균', '고급성_중앙값','쾌적성_중앙값','감성_중앙값',
    '쾌적성_평균', '감성_평균'
]
df[image_cols] = df[image_cols].replace(0, np.nan)
print("ℹ️ 이미지 변수 내 0 -> NaN 변환 완료!")

drop_cols = ['카테고리', '매장명', '블루리본 여부','고급성_중앙값','쾌적성_중앙값','감성_중앙값',
            'idx_family_weighted_score']
X_base = df.drop(columns=drop_cols)

print(f"✅ 사용된 총 변수 개수: {len(X_base.columns)}개")

# =========================================================================
# 2. Train / Test 분리
# =========================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42, stratify=y
)

# =========================================================================
# 3. 조합 및 CV 설정
# =========================================================================
corr_pairs = [
    ("idx_anniversary_weighted_score", "idx_luxury_weighted_score"),
    ("idx_date_weighted_score", "idx_ambiance_weighted_score"),
    ("고급성_평균", "감성_평균"),
    ("idx_service_weighted_score", "idx_luxury_weighted_score")
]

N_TRIALS = 20  # 두 모델을 돌리므로 시간을 위해 20으로 단축 (필요시 50으로 수정)
CV_SPLITS = 5

cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=42)
all_combinations = list(itertools.product([0, 1], repeat=len(corr_pairs)))

print(f"\n✅ 총 실험 조합 개수: {len(all_combinations)}개 (LGBM, XGB 각각 최적화 진행)")

best_lgbm_score = 0
best_lgbm_params = None
best_lgbm_drop_cols = None

best_xgb_score = 0
best_xgb_params = None
best_xgb_drop_cols = None

# =========================================================================
# 4. 각 모델별 최적의 변수 조합 및 하이퍼파라미터 찾기
# =========================================================================
optuna.logging.set_verbosity(optuna.logging.WARNING) # Optuna 로그 숨기기 (진행상황만 출력)

for combo_idx, combo in enumerate(all_combinations, start=1):
    print(f"\n[ 🔄 조합 {combo_idx}/{len(all_combinations)} 탐색 중... ]")
    
    selected_drop_cols = list(dict.fromkeys([var1 if choice == 0 else var2 for choice, (var1, var2) in zip(combo, corr_pairs)]))
    existing_drop_cols = [col for col in selected_drop_cols if col in X_train.columns]
    
    X_train_sub = X_train.drop(columns=existing_drop_cols)
    
    # --- LightGBM 목적 함수 ---
    def objective_lgbm(trial):
        params = {
            "objective": "binary", "metric": "binary_logloss", "random_state": 42, "verbose": -1, "n_jobs": -1,
            "n_estimators": trial.suggest_int("n_estimators", 100, 300),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 10, 31),
            "max_depth": trial.suggest_int("max_depth", 3, 7),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 50),
            "subsample": trial.suggest_float("subsample", 0.5, 0.9),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.9)
        }
        model = lgb.LGBMClassifier(**params)
        # 윈도우 프리징 방지를 위해 n_jobs=1 유지
        return np.mean(cross_val_score(model, X_train_sub, y_train, cv=cv, scoring="roc_auc", n_jobs=1))

    # --- XGBoost 목적 함수 ---
    def objective_xgb(trial):
        params = {
            "objective": "binary:logistic", "eval_metric": "logloss", "random_state": 42, "n_jobs": -1, "tree_method": "hist",
            "n_estimators": trial.suggest_int("n_estimators", 100, 300),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            "max_depth": trial.suggest_int("max_depth", 2, 6),
            "min_child_weight": trial.suggest_int("min_child_weight", 3, 10),
            "subsample": trial.suggest_float("subsample", 0.5, 0.9),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.9)
        }
        model = xgb.XGBClassifier(**params)
        # 윈도우 프리징 방지를 위해 n_jobs=1 유지
        return np.mean(cross_val_score(model, X_train_sub, y_train, cv=cv, scoring="roc_auc", n_jobs=1))

    # 튜닝 실행
    study_lgbm = optuna.create_study(direction="maximize")
    study_lgbm.optimize(objective_lgbm, n_trials=N_TRIALS)
    
    study_xgb = optuna.create_study(direction="maximize")
    study_xgb.optimize(objective_xgb, n_trials=N_TRIALS)
    
    # 최고 점수 갱신 시 기록
    if study_lgbm.best_value > best_lgbm_score:
        best_lgbm_score = study_lgbm.best_value
        best_lgbm_params = study_lgbm.best_params
        best_lgbm_drop_cols = existing_drop_cols
        
    if study_xgb.best_value > best_xgb_score:
        best_xgb_score = study_xgb.best_value
        best_xgb_params = study_xgb.best_params
        best_xgb_drop_cols = existing_drop_cols


# =========================================================================
# 5. 최종 앙상블 모델 학습 및 예측
# =========================================================================
print("\n" + "=" * 90)
print(f"🥇 최고 성능 LightGBM CV AUC: {best_lgbm_score:.4f}")
print(f"🥇 최고 성능 XGBoost CV AUC : {best_xgb_score:.4f}")
print("=" * 90)

# LGBM 최종 학습
X_train_lgbm = X_train.drop(columns=best_lgbm_drop_cols)
X_test_lgbm = X_test.drop(columns=best_lgbm_drop_cols)

final_lgbm = lgb.LGBMClassifier(**best_lgbm_params, objective="binary", random_state=42, verbose=-1, n_jobs=-1)
final_lgbm.fit(X_train_lgbm, y_train)

# XGB 최종 학습
X_train_xgb = X_train.drop(columns=best_xgb_drop_cols)
X_test_xgb = X_test.drop(columns=best_xgb_drop_cols)

final_xgb = xgb.XGBClassifier(**best_xgb_params, objective="binary:logistic", random_state=42, n_jobs=-1, tree_method="hist")
final_xgb.fit(X_train_xgb, y_train)

# 각 모델의 예측 확률 계산 (Train / Test)
train_proba_lgbm = final_lgbm.predict_proba(X_train_lgbm)[:, 1]
train_proba_xgb = final_xgb.predict_proba(X_train_xgb)[:, 1]
test_proba_lgbm = final_lgbm.predict_proba(X_test_lgbm)[:, 1]
test_proba_xgb = final_xgb.predict_proba(X_test_xgb)[:, 1]

# 🤝 소프트 보팅 (Soft Voting) : 두 모델의 예측 확률을 5:5로 섞음
train_proba_ensemble = (train_proba_lgbm + train_proba_xgb) / 2
test_proba_ensemble = (test_proba_lgbm + test_proba_xgb) / 2

# =========================================================================
# 6. 최적 임계값(Threshold) 찾기 및 평가
# =========================================================================

# Train 데이터 기준으로 F1 스코어를 가장 높여주는 Threshold 찾기
precisions, recalls, thresholds = precision_recall_curve(y_train, train_proba_ensemble)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_threshold = thresholds[np.argmax(f1_scores)]

print(f"\n💡 앙상블 모델 최적 예측 기준점(Threshold): {best_threshold:.4f} (기본 0.5에서 조정됨)")

# 찾은 Threshold를 Test 데이터에 적용하여 0 또는 1로 분류
y_train_pred_ens = (train_proba_ensemble >= best_threshold).astype(int)
y_test_pred_ens = (test_proba_ensemble >= best_threshold).astype(int)

# 평가지표 산출
train_auc_ens = roc_auc_score(y_train, train_proba_ensemble)
test_auc_ens = roc_auc_score(y_test, test_proba_ensemble)
auc_gap = abs(train_auc_ens - test_auc_ens)

print("\n[ 🎯 앙상블 모델 Train vs Test 과적합 검증 ]")
print(f"Train ROC-AUC : {train_auc_ens:.4f}")
print(f"Test ROC-AUC  : {test_auc_ens:.4f}")
print(f"AUC Gap       : {auc_gap:.4f} (작을수록 안정적)")

print("\n[ 📊 앙상블 모델 최종 Test 실전 성능 ]")
print(f"Accuracy  : {accuracy_score(y_test, y_test_pred_ens):.4f}")
print(f"Precision : {precision_score(y_test, y_test_pred_ens):.4f}")
print(f"Recall    : {recall_score(y_test, y_test_pred_ens):.4f}")
print(f"F1-score  : {f1_score(y_test, y_test_pred_ens):.4f}")

print("\n[ 상세 분류 리포트 ]")
print(classification_report(y_test, y_test_pred_ens, zero_division=0))

# =========================================================================
# 7. 예측 확률 시각화 (선택사항)
# =========================================================================
plt.figure(figsize=(10, 6))
sns.kdeplot(test_proba_lgbm, label='LightGBM 확률', fill=True, color='skyblue', alpha=0.3)
sns.kdeplot(test_proba_xgb, label='XGBoost 확률', fill=True, color='lightgreen', alpha=0.3)
sns.kdeplot(test_proba_ensemble, label='앙상블(Soft Voting) 확률', fill=True, color='coral', alpha=0.6)

plt.axvline(best_threshold, color='red', linestyle='--', label=f'최적 Threshold ({best_threshold:.2f})')
plt.title("모델별 예측 확률 분포 비교 및 앙상블 효과")
plt.xlabel("블루리본 선정 예측 확률")
plt.ylabel("밀도")
plt.legend()
plt.tight_layout()
plt.show()

### 앙상블 수정
선택 가중치 스스로 찾도록

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import optuna
import itertools

from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    classification_report, roc_auc_score, f1_score, precision_recall_curve
)

import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rc('font', family='Malgun Gothic') 
plt.rcParams['axes.unicode_minus'] = False

print("\n[ 🚀 LightGBM + XGBoost 앙상블 (소프트 보팅) 최종 모델링 ]")

# =========================================================================
# 1. 데이터 불러오기 및 X, y 세팅
# =========================================================================
df = pd.read_csv('../data/블루리본_최최종_마참내_v1.2.csv', encoding='cp949')
y = df['블루리본 여부']

image_cols = [
    '고급성_평균', '고급성_중앙값','쾌적성_중앙값','감성_중앙값',
    '쾌적성_평균', '감성_평균'
]
df[image_cols] = df[image_cols].replace(0, np.nan)
print("ℹ️ 이미지 변수 내 0 -> NaN 변환 완료!")

drop_cols = ['카테고리', '매장명', '블루리본 여부','고급성_중앙값','쾌적성_중앙값','감성_중앙값',
            'idx_family_weighted_score']
X_base = df.drop(columns=drop_cols)

print(f"✅ 사용된 총 변수 개수: {len(X_base.columns)}개")

# =========================================================================
# 2. Train / Test 분리
# =========================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42, stratify=y
)

# =========================================================================
# 3. 조합 및 CV 설정
# =========================================================================
corr_pairs = [
    ("idx_anniversary_weighted_score", "idx_luxury_weighted_score"),
    ("idx_date_weighted_score", "idx_ambiance_weighted_score"),
    ("고급성_평균", "감성_평균"),
    ("idx_service_weighted_score", "idx_luxury_weighted_score")
]

N_TRIALS = 20  # 두 모델을 돌리므로 시간을 위해 20으로 단축 (필요시 50으로 수정)
CV_SPLITS = 5

cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=42)
all_combinations = list(itertools.product([0, 1], repeat=len(corr_pairs)))

print(f"\n✅ 총 실험 조합 개수: {len(all_combinations)}개 (LGBM, XGB 각각 최적화 진행)")

best_lgbm_score = 0
best_lgbm_params = None
best_lgbm_drop_cols = None

best_xgb_score = 0
best_xgb_params = None
best_xgb_drop_cols = None

# =========================================================================
# 4. 각 모델별 최적의 변수 조합 및 하이퍼파라미터 찾기
# =========================================================================
optuna.logging.set_verbosity(optuna.logging.WARNING) # Optuna 로그 숨기기 (진행상황만 출력)

for combo_idx, combo in enumerate(all_combinations, start=1):
    print(f"\n[ 🔄 조합 {combo_idx}/{len(all_combinations)} 탐색 중... ]")
    
    selected_drop_cols = list(dict.fromkeys([var1 if choice == 0 else var2 for choice, (var1, var2) in zip(combo, corr_pairs)]))
    existing_drop_cols = [col for col in selected_drop_cols if col in X_train.columns]
    
    X_train_sub = X_train.drop(columns=existing_drop_cols)
    
    # --- LightGBM 목적 함수 ---
    def objective_lgbm(trial):
        params = {
            "objective": "binary", "metric": "binary_logloss", "random_state": 42, "verbose": -1, "n_jobs": -1,
            "n_estimators": trial.suggest_int("n_estimators", 100, 300),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 10, 31),
            "max_depth": trial.suggest_int("max_depth", 3, 7),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 50),
            "subsample": trial.suggest_float("subsample", 0.5, 0.9),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.9)
        }
        model = lgb.LGBMClassifier(**params)
        # 윈도우 프리징 방지를 위해 n_jobs=1 유지
        return np.mean(cross_val_score(model, X_train_sub, y_train, cv=cv, scoring="roc_auc", n_jobs=1))

    # --- XGBoost 목적 함수 ---
    def objective_xgb(trial):
        params = {
            "objective": "binary:logistic", "eval_metric": "logloss", "random_state": 42, "n_jobs": -1, "tree_method": "hist",
            "n_estimators": trial.suggest_int("n_estimators", 100, 300),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            "max_depth": trial.suggest_int("max_depth", 2, 6),
            "min_child_weight": trial.suggest_int("min_child_weight", 3, 10),
            "subsample": trial.suggest_float("subsample", 0.5, 0.9),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.9)
        }
        model = xgb.XGBClassifier(**params)
        # 윈도우 프리징 방지를 위해 n_jobs=1 유지
        return np.mean(cross_val_score(model, X_train_sub, y_train, cv=cv, scoring="roc_auc", n_jobs=1))

    # 튜닝 실행
    study_lgbm = optuna.create_study(direction="maximize")
    study_lgbm.optimize(objective_lgbm, n_trials=N_TRIALS)
    
    study_xgb = optuna.create_study(direction="maximize")
    study_xgb.optimize(objective_xgb, n_trials=N_TRIALS)
    
    # 최고 점수 갱신 시 기록
    if study_lgbm.best_value > best_lgbm_score:
        best_lgbm_score = study_lgbm.best_value
        best_lgbm_params = study_lgbm.best_params
        best_lgbm_drop_cols = existing_drop_cols
        
    if study_xgb.best_value > best_xgb_score:
        best_xgb_score = study_xgb.best_value
        best_xgb_params = study_xgb.best_params
        best_xgb_drop_cols = existing_drop_cols


# =========================================================================
# 5. 최종 앙상블 모델 학습 및 예측
# =========================================================================
print("\n" + "=" * 90)
print(f"🥇 최고 성능 LightGBM CV AUC: {best_lgbm_score:.4f}")
print(f"🥇 최고 성능 XGBoost CV AUC : {best_xgb_score:.4f}")
print("=" * 90)

# LGBM 최종 학습
X_train_lgbm = X_train.drop(columns=best_lgbm_drop_cols)
X_test_lgbm = X_test.drop(columns=best_lgbm_drop_cols)

final_lgbm = lgb.LGBMClassifier(**best_lgbm_params, objective="binary", random_state=42, verbose=-1, n_jobs=-1)
final_lgbm.fit(X_train_lgbm, y_train)

# XGB 최종 학습
X_train_xgb = X_train.drop(columns=best_xgb_drop_cols)
X_test_xgb = X_test.drop(columns=best_xgb_drop_cols)

final_xgb = xgb.XGBClassifier(**best_xgb_params, objective="binary:logistic", random_state=42, n_jobs=-1, tree_method="hist")
final_xgb.fit(X_train_xgb, y_train)

# 각 모델의 예측 확률 계산 (Train / Test)
train_proba_lgbm = final_lgbm.predict_proba(X_train_lgbm)[:, 1]
train_proba_xgb = final_xgb.predict_proba(X_train_xgb)[:, 1]
test_proba_lgbm = final_lgbm.predict_proba(X_test_lgbm)[:, 1]
test_proba_xgb = final_xgb.predict_proba(X_test_xgb)[:, 1]

# =========================================================================
# 6. 최적 가중치(Weight) 및 임계값(Threshold) 찾기
# =========================================================================
print("\n[ 🔍 최적의 앙상블 가중치 비율 및 임계값 탐색 중... ]")

best_weight_lgbm = 0.5
best_threshold = 0.5
best_train_f1 = 0

# LightGBM 가중치를 0.0부터 1.0까지 0.05 단위로 테스트 (총 21개 비율 테스트)
weights = np.linspace(0.0, 1.0, 21)

for w_lgbm in weights:
    w_xgb = 1.0 - w_lgbm
    
    # 현재 비율로 예측 확률 계산
    temp_train_proba = (train_proba_lgbm * w_lgbm) + (train_proba_xgb * w_xgb)
    
    # 현재 섞인 확률 분포에서 최적의 Threshold 찾기 (Train 기준)
    precisions, recalls, thresholds = precision_recall_curve(y_train, temp_train_proba)
    # thresholds 길이와 맞추기 위해 마지막 값 제외
    f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-10)
    
    if len(f1_scores) > 0:
        max_f1_idx = np.argmax(f1_scores)
        temp_best_f1 = f1_scores[max_f1_idx]
        temp_best_thresh = thresholds[max_f1_idx]
        
        # 최고 Train F1 갱신 시 기록
        if temp_best_f1 > best_train_f1:
            best_train_f1 = temp_best_f1
            best_weight_lgbm = w_lgbm
            best_threshold = temp_best_thresh

best_weight_xgb = 1.0 - best_weight_lgbm

print(f"🥇 최적 앙상블 비율 - LightGBM: {best_weight_lgbm*100:.0f}%, XGBoost: {best_weight_xgb*100:.0f}%")

print(f"\n💡 앙상블 모델 최적 예측 기준점(Threshold): {best_threshold:.4f} (기본 0.5에서 조정됨)")

# 🤝 찾아낸 최적 가중치로 최종 예측 확률 고정
train_proba_ensemble = (train_proba_lgbm * best_weight_lgbm) + (train_proba_xgb * best_weight_xgb)
test_proba_ensemble = (test_proba_lgbm * best_weight_lgbm) + (test_proba_xgb * best_weight_xgb)

# 찾은 Threshold를 Test 데이터에 적용하여 0 또는 1로 분류
y_train_pred_ens = (train_proba_ensemble >= best_threshold).astype(int)
y_test_pred_ens = (test_proba_ensemble >= best_threshold).astype(int)

# 평가지표 산출
train_auc_ens = roc_auc_score(y_train, train_proba_ensemble)
test_auc_ens = roc_auc_score(y_test, test_proba_ensemble)
auc_gap = abs(train_auc_ens - test_auc_ens)

print("\n[ 🎯 앙상블 모델 Train vs Test 과적합 검증 ]")
print(f"Train ROC-AUC : {train_auc_ens:.4f}")
print(f"Test ROC-AUC  : {test_auc_ens:.4f}")
print(f"AUC Gap       : {auc_gap:.4f} (작을수록 안정적)")

print("\n[ 📊 앙상블 모델 최종 Test 실전 성능 ]")
print(f"Accuracy  : {accuracy_score(y_test, y_test_pred_ens):.4f}")
print(f"Precision : {precision_score(y_test, y_test_pred_ens):.4f}")
print(f"Recall    : {recall_score(y_test, y_test_pred_ens):.4f}")
print(f"F1-score  : {f1_score(y_test, y_test_pred_ens):.4f}")

print("\n[ 상세 분류 리포트 ]")
print(classification_report(y_test, y_test_pred_ens, zero_division=0))

# =========================================================================
# 7. 예측 확률 시각화 (선택사항)
# =========================================================================
plt.figure(figsize=(10, 6))
sns.kdeplot(test_proba_lgbm, label='LightGBM 확률', fill=True, color='skyblue', alpha=0.3)
sns.kdeplot(test_proba_xgb, label='XGBoost 확률', fill=True, color='lightgreen', alpha=0.3)
sns.kdeplot(test_proba_ensemble, label=f'앙상블({best_weight_lgbm*100:.0f}:{best_weight_xgb*100:.0f}) 확률', fill=True, color='coral', alpha=0.6)

plt.axvline(best_threshold, color='red', linestyle='--', label=f'최적 Threshold ({best_threshold:.2f})')
plt.title("모델별 예측 확률 분포 비교 및 앙상블 효과")
plt.xlabel("블루리본 선정 예측 확률")
plt.ylabel("밀도")
plt.legend()
plt.tight_layout()
plt.show()

### ml_lgbm_optuna 전체 변수로 다시 수정

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    classification_report, roc_auc_score, f1_score
)
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 설정
plt.rc('font', family='Malgun Gothic') 
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
print("\n[ 🚀 LightGBM + Optuna를 활용한 블루리본 예측 모델링 ]")

# =========================================================================
# 1. 데이터 불러오기 및 X, y 세팅
# =========================================================================
df = pd.read_csv('../data/블루리본_최최종_마참내_v1.2.csv', encoding='cp949')
y = df['블루리본 여부']

# 이미지 모델 결과 컬럼들의 0 값을 NaN으로 변경 (LightGBM 특화 처리)
image_cols = [
    '고급성_평균', '고급성_중앙값', 
    '쾌적성_평균', '쾌적성_중앙값', 
    '감성_평균', '감성_중앙값'
]
df[image_cols] = df[image_cols].replace(0, np.nan)
print("ℹ️ 이미지 변수 내 0 -> NaN 변환 완료!")


In [ ]:
# --- 변수 제거 설정 ---
# 기본적으로 제거할 변수
base_drop_cols = ['카테고리', '매장명', '블루리본 여부', 'idx_family_weighted_score']

final_drop_cols = list(set(base_drop_cols))
X_base = df.drop(columns=final_drop_cols)

print("\n✅ 다음 변수들을 제거하고 모델링을 시작합니다:")
for col in sorted(final_drop_cols):
    if col in df.columns:
        print(f" - {col}")
print(f"✅ 사용된 총 변수 개수: {len(X_base.columns)}개")


In [ ]:
# =========================================================================
# 2. Train / Test 분리
# =========================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# =========================================================================
# 3. Optuna를 사용한 하이퍼파라미터 튜닝
# =========================================================================
print("\n[ 🔎 Optuna를 사용하여 최적의 하이퍼파라미터 탐색 시작 ]")

N_TRIALS = 100      # 탐색 횟수 (시간이 오래 걸리면 50으로 줄여서 테스트)
CV_SPLITS = 5

cv = StratifiedKFold(
    n_splits=CV_SPLITS,
    shuffle=True,
    random_state=42
)

def objective_lgbm_regularized(trial):
    params = {
        # 기본 설정
        "objective": "binary",
        "metric": "binary_logloss",
        "boosting_type": "gbdt",
        "random_state": 42,
        "verbose": -1,
        "n_jobs": -1,

        # 모델 복잡도 제어
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 7, 31),
        "max_depth": trial.suggest_int("max_depth", 2, 6),

        # 과적합 방지용 샘플링
        "subsample": trial.suggest_float("subsample", 0.5, 0.85),
        "subsample_freq": trial.suggest_int("subsample_freq", 1, 5),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.85),

        # 리프 최소 데이터 수
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 50),
        "min_child_weight": trial.suggest_float("min_child_weight", 0.001, 10.0, log=True),

        # 규제
        "reg_alpha": trial.suggest_float("reg_alpha", 0.01, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.01, 10.0, log=True),

        # 분할 이득 제한
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0)
    }
    
    model = lgb.LGBMClassifier(**params)
    
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )
    
    return np.mean(scores)

study = optuna.create_study(
    direction="maximize",
    study_name="LGBM_Optimization"
)

study.optimize(
    objective_lgbm_regularized,
    n_trials=N_TRIALS,
    show_progress_bar=True
)

best_params = study.best_params

print("\n" + "=" * 90)
print("🏆 Optuna 튜닝 완료")
print(f"Best CV ROC-AUC: {study.best_value:.4f}")
print("\nBest Hyperparameters:")
print(best_params)
print("=" * 90)


In [ ]:
# =========================================================================
# 4. 최적 파라미터로 최종 모델 학습 및 평가
# =========================================================================

final_model = lgb.LGBMClassifier(
    **best_params,
    objective="binary",
    metric="binary_logloss",
    boosting_type="gbdt",
    random_state=42,
    verbose=-1,
    n_jobs=-1
)

final_model.fit(X_train, y_train)

y_train_pred = final_model.predict(X_train)
y_test_pred = final_model.predict(X_test)

y_train_proba = final_model.predict_proba(X_train)[:, 1]
y_test_proba = final_model.predict_proba(X_test)[:, 1]

train_auc = roc_auc_score(y_train, y_train_proba)
test_auc = roc_auc_score(y_test, y_test_proba)
auc_gap = abs(train_auc - test_auc)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print("\n[ 🎯 Train vs Test 과적합 검증 ]")
print(f"Train ROC-AUC : {train_auc:.4f}")
print(f"Test ROC-AUC  : {test_auc:.4f}")
print(f"AUC Gap       : {auc_gap:.4f} (작을수록 안정적)")
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")

print("\n[ 📊 최종 Test 실전 성능 ]")
print(f"Accuracy  : {test_acc:.4f}")
print(classification_report(y_test, y_test_pred, zero_division=0))


In [ ]:
importance_df

In [ ]:
# =========================================================================
# 5. 특성 중요도 (Feature Importance) 확인
# =========================================================================
importance_gain = final_model.booster_.feature_importance(importance_type='gain')

importance_df = pd.DataFrame({
    'Feature': X_train.columns, 
    'Importance': importance_gain
}).sort_values('Importance', ascending=False).head(15)

print("\n[ 🔥 상위 15개 중요 변수 ]")
print(importance_df.to_string(index=False))

# 시각화
fig, ax = plt.subplots(figsize=(12, 8))
lgb.plot_importance(final_model, ax=ax, max_num_features=20, height=0.8, 
                    importance_type='gain', title='LightGBM Feature Importance (Gain)')
plt.tight_layout()
plt.show()

### ml_lgbm_optuna 파라미터 튜닝 v1.0

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    classification_report, roc_auc_score, f1_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import optuna.visualization as vis

# 한글 폰트 설정
plt.rc('font', family='Malgun Gothic') 
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
print("\n[ 🚀 LightGBM + Optuna를 활용한 블루리본 예측 모델링 ]")

# =========================================================================
# 1. 데이터 불러오기 및 X, y 세팅
# =========================================================================
df = pd.read_csv('../data/블루리본_최최종_마참내_v1.2.csv', encoding='cp949')
y = df['블루리본 여부']

# 이미지 모델 결과 컬럼들의 0 값을 NaN으로 변경 (LightGBM 특화 처리)
image_cols = ['고급성_평균', '쾌적성_평균', '감성_평균']
df[image_cols] = df[image_cols].replace(0, np.nan)
print("ℹ️ 이미지 변수 내 0 -> NaN 변환 완료!")

# 'has_missing' 이라는 새로운 파생 변수 생성
# 결측값이 하나라도 있으면 1, 완전한 데이터면 0으로 채워집니다.
df['has_missing'] = df.isnull().any(axis=1).astype(int)

# 💡 결과가 잘 들어갔는지 확인해보기
print(df['has_missing'].value_counts())

# '카테고리' 컬럼을 원-핫 인코딩합니다.
# dtype=int를 넣어야 True/False가 아닌 1/0 형태로 깔끔하게 들어갑니다.
df = pd.get_dummies(df, columns=['카테고리'], prefix='카테고리', dtype=int)

In [ ]:
# --- 변수 제거 설정 ---
# 기본적으로 제거할 변수
base_drop_cols = ['매장명', '블루리본 여부', 'idx_family_weighted_score',
                 '고급성_중앙값', '쾌적성_중앙값', '감성_중앙값']

final_drop_cols = list(set(base_drop_cols))
X_base = df.drop(columns=final_drop_cols)

print("\n✅ 다음 변수들을 제거하고 모델링을 시작합니다:")
for col in sorted(final_drop_cols):
    if col in df.columns:
        print(f" - {col}")
print(f"✅ 사용된 총 변수 개수: {len(X_base.columns)}개")


In [ ]:
# =========================================================================
# 2. Train / Test 분리
# =========================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42, stratify=y
)

- n_estimators : 150 ~ 400 -> 100 ~ 250
- learning_rate : 0.01 ~ 0.05 -> 0.01 ~ 0.05
- num_leaves (최대 리프 수): 10 ~ 20 ➔ 3 ~ 7
- max_depth (최대 깊이): 3 ~ 5 ➔ 2 ~ 3

- subsample: 0.65 ~ 0.85 ➔ 0.60 ~ 0.80
- subsample_freq: 1 ~ 3 ➔ 1 ~ 3
- colsample_bytree: 0.6 ~ 0.8 ➔ 0.50 ~ 0.80
  
- min_child_samples: 30 ~ 50 ➔ 50 ~ 80
- min_child_weight: 0.1 ~ 5.0 ➔ 0.1 ~ 5.0

- reg_alpha / reg_lambda (규제): 1.0 ~ 4.0 ➔ 2.0 ~ 10.0

- min_split_gain: 0.1 ~ 0.5 ➔ 0.1 ~ 0.5

- scoring: roc_auc ➔ f1

In [ ]:
# =========================================================================
# 3. Optuna를 사용한 하이퍼파라미터 튜닝
# =========================================================================
print("\n[ 🔎 Optuna를 사용하여 최적의 하이퍼파라미터 탐색 시작 ]")

N_TRIALS = 100      # 탐색 횟수 (시간이 오래 걸리면 50으로 줄여서 테스트)
CV_SPLITS = 5

cv = StratifiedKFold(
    n_splits=CV_SPLITS,
    shuffle=True,
    random_state=42
)

def objective_lgbm_regularized(trial):
    max_depth = trial.suggest_int("max_depth", 2, 4)
    max_leaves = min(10, 2 ** max_depth)
    num_leaves = trial.suggest_int("num_leaves", 3, max_leaves)

    params = {
        "objective": "binary",
        "metric": "binary_logloss",
        "boosting_type": "gbdt",
        "random_state": 42,
        "verbose": -1,
        "n_jobs": -1,

        # 속도 및 반복 설정
        "n_estimators": trial.suggest_int("n_estimators", 150, 500), # 조금 더 열어두되 조기종료로 컷
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.05, log=True),
        "max_depth": max_depth,
        "num_leaves": num_leaves,

        # 2. 과적합 방지용 샘플링 (매우 좋은 설정 유지)
        "subsample": trial.suggest_float("subsample", 0.65, 0.85),
        "subsample_freq": trial.suggest_int("subsample_freq", 2, 4),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.50, 0.70),

        # 3. 리프 조건 규제 강화 (격차 감소 목적)
        "min_child_samples": trial.suggest_int("min_child_samples", 30, 100), # 하한/상한 상향
        "min_child_weight": trial.suggest_float("min_child_weight", 0.5, 10.0, log=True),

        # 4. 강한 L1, L2 규제
        "reg_alpha": trial.suggest_float("reg_alpha", 1.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 5.0),

        # 5. 분할 이득 허들
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 0.5)
    }
    
    # params = {
    #     # 기본 설정
    #     "objective": "binary",
    #     "metric": "binary_logloss",
    #     "boosting_type": "gbdt",
    #     "random_state": 42,
    #     "verbose": -1,
    #     "n_jobs": -1,

    #     # 1. 모델 복잡도 (논리적 정렬 및 상한선 축소)
    #     "n_estimators": trial.suggest_int("n_estimators", 150, 400), # 불필요한 반복 학습 차단
    #     "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.05, log=True),
    #     "max_depth": trial.suggest_int("max_depth", 3, 5), # 깊이를 보수적으로 제한
    #     "num_leaves": trial.suggest_int("num_leaves", 10, 20), # 💡 max_depth에 맞춰 최대 15로 제한

    #     # 2. 과적합 방지용 샘플링
    #     "subsample": trial.suggest_float("subsample", 0.65, 0.85),
    #     "subsample_freq": trial.suggest_int("subsample_freq", 1, 3),
    #     "colsample_bytree": trial.suggest_float("colsample_bytree", 0.60, 0.80),

    #     # 3. 리프 최소 데이터 수 (하한선 상향)
    #     "min_child_samples": trial.suggest_int("min_child_samples", 30, 50), # 💡 조금 더 깐깐하게
    #     "min_child_weight": trial.suggest_float("min_child_weight", 0.1, 5.0, log=True),

    #     # 4. 규제 (현재 잘 작동하고 있으므로 범위만 최적화)
    #     "reg_alpha": trial.suggest_float("reg_alpha", 1.0, 4.0),
    #     "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 4.0),

    #     # 5. 분할 이득 허들
    #     "min_split_gain": trial.suggest_float("min_split_gain", 0.1, 0.5)
    # }

    model = lgb.LGBMClassifier(**params)
    
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )
    
    return np.mean(scores)

study = optuna.create_study(
    direction="maximize",
    study_name="LGBM_Optimization"
)

study.optimize(
    objective_lgbm_regularized,
    n_trials=N_TRIALS,
    show_progress_bar=True
)

best_params = study.best_params

print("\n" + "=" * 90)
print("🏆 Optuna 튜닝 완료")
print(f"Best CV ROC-AUC: {study.best_value:.4f}")
print("\nBest Hyperparameters:")
print(best_params)
print("=" * 90)

# print("\n[ 📊 Optuna 시각화 그래프 생성 중... ]")

# # 1. 최적화 탐색 이력 (점수가 어떻게 올라갔는지 확인)
# fig_history = vis.plot_optimization_history(study)
# fig_history.show()

# # 2. 하이퍼파라미터 중요도 (어떤 변수가 성능에 가장 큰 영향을 줬는지 확인)
# fig_importance = vis.plot_param_importances(study)
# fig_importance.show()

# # 3. 하이퍼파라미터 평행 좌표계 (성능이 좋은 파라미터들의 조합 확인)
# fig_parallel = vis.plot_parallel_coordinate(study)
# fig_parallel.show()

# # 4. 개별 파라미터 변화에 따른 성능 산점도 (선택 사항)
# fig_slice = vis.plot_slice(study)
# fig_slice.show()

🎀 블루리본 매장 추천 (현재 상황): 사용자에게 "여긴 무조건 맛집!"이라고 추천해야 한다면, 기준을 0.5에서 0.6이나 0.65로 높이는 것이 합리적입니다. 51%의 애매한 확률을 가진 식당을 추천했다가 사용자가 실망하는 위험을 줄이기 위해서입니다

##### threshold 통합 버전

In [ ]:
# 사용자가 직접 지정한 고정 파라미터 
best_params = {
    "n_estimators": 176,
    "learning_rate": 0.0167,
    "max_depth": 4,
    "num_leaves": 19,
    "subsample": 0.723,          # 72.3%
    "subsample_freq": 1,
    "colsample_bytree": 0.628,   # 62.8%
    "min_child_samples": 35,
    "min_child_weight": 0.4544,
    "reg_alpha": 2.401,          # L1 규제
    "reg_lambda": 2.557,         # L2 규제
    "min_split_gain": 0.174
}

In [ ]:
# =========================================================================
# 4. 최적 파라미터로 최종 모델 학습 
# =========================================================================
print("\n[ 🚀 최종 모델 학습 시작 ]")

final_model = lgb.LGBMClassifier(
    **best_params, # Optuna에서 찾은 최적 파라미터
    objective="binary",
    metric="binary_logloss",
    boosting_type="gbdt",
    random_state=42,
    verbose=-1,
    n_jobs=-1
)

final_model.fit(X_train, y_train)

# 확률값 계산 (AUC 계산 및 Threshold 조정에 공통으로 사용)
y_train_proba = final_model.predict_proba(X_train)[:, 1]
y_test_proba = final_model.predict_proba(X_test)[:, 1]

# =========================================================================
# 5. 모델 기본 체력 및 과적합 검증 (AUC 중심)
# =========================================================================
train_auc = roc_auc_score(y_train, y_train_proba)
test_auc = roc_auc_score(y_test, y_test_proba)
auc_gap = abs(train_auc - test_auc)

# 기본 임계값(0.5) 기준의 정확도
train_acc_default = accuracy_score(y_train, (y_train_proba >= 0.5).astype(int))
test_acc_default = accuracy_score(y_test, (y_test_proba >= 0.5).astype(int))

print("\n[ 🎯 Train vs Test 과적합 검증 (Default Threshold: 0.5) ]")
print(f"Train ROC-AUC : {train_auc:.4f}")
print(f"Test ROC-AUC  : {test_auc:.4f}")
print(f"AUC Gap       : {auc_gap:.4f} (0.05 이하 권장)")
print(f"Train Accuracy: {train_acc_default:.4f}")
print(f"Test Accuracy : {test_acc_default:.4f}")

# =========================================================================
# 6. 비즈니스 최적화를 위한 Threshold 탐색
# =========================================================================
print("\n[ 🔍 다양한 Threshold에 따른 실전 성능 변화 테스트 ]")
print("Threshold | Accuracy | Precision | Recall | F1-Score")
print("-" * 55)

for threshold in np.arange(0.50, 0.72, 0.02):
    temp_pred = (y_test_proba >= threshold).astype(int)
    
    acc = accuracy_score(y_test, temp_pred)
    prec = precision_score(y_test, temp_pred, zero_division=0)
    rec = recall_score(y_test, temp_pred)
    f1 = f1_score(y_test, temp_pred)
    
    print(f"   {threshold:.2f}   |  {acc:.4f}  |   {prec:.4f}  | {rec:.4f} |  {f1:.4f}")

In [ ]:
# =========================================================================
# 7. 최종 Threshold 적용 및 상세 리포트 출력
# =========================================================================
# 💡 위 루프 결과를 확인한 후, 가장 마음에 드는 숫자를 아래에 입력하세요!
FINAL_THRESHOLD = 0.52

# Train과 Test 각각에 새로운 Threshold 적용
final_train_pred = (y_train_proba >= FINAL_THRESHOLD).astype(int)
final_test_pred = (y_test_proba >= FINAL_THRESHOLD).astype(int)

print(f"\n[ 📊 최종 실전 성능 (적용된 Threshold: {FINAL_THRESHOLD}) ]")
# 💡 최종 Train 정확도 출력 추가
print(f"최종 Train Accuracy : {accuracy_score(y_train, final_train_pred):.4f}")
print(f"최종 Test Accuracy  : {accuracy_score(y_test, final_test_pred):.4f}")

# ✨ Train 데이터의 Precision, Recall, F1-score 출력 추가
print("\n[ Train Data Classification Report ]")
print(classification_report(y_train, final_train_pred, zero_division=0, digits=4))

# 최종 Accuracy Gap 확인 (선택 사항)
print(f"Accuracy Gap        : {abs(accuracy_score(y_train, final_train_pred) - accuracy_score(y_test, final_test_pred)):.4f}")
print("-" * 55)
print(classification_report(y_test, final_test_pred, zero_division=0, digits=4))

In [ ]:
# =========================================================================
# 5. 특성 중요도 (Feature Importance) 확인
# =========================================================================
importance_gain = final_model.booster_.feature_importance(importance_type='gain')

importance_df = pd.DataFrame({
    'Feature': X_train.columns, 
    'Importance': importance_gain
}).sort_values('Importance', ascending=False).head(15)

# print("\n[ 🔥 상위 15개 중요 변수 ]")
# print(importance_df.to_string(index=False))

# 시각화
fig, ax = plt.subplots(figsize=(12, 8))
lgb.plot_importance(final_model, ax=ax, max_num_features=20, height=0.8, 
                    importance_type='gain', title='LightGBM Feature Importance (Gain)')
plt.tight_layout()
plt.show()

In [ ]:
importance_df

### 베이지안

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    classification_report, roc_auc_score, f1_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import optuna.visualization as vis

# 한글 폰트 설정
plt.rc('font', family='Malgun Gothic') 
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
all_df = pd.read_csv('../data/리뷰_감정분석.csv')
all_df.head()
all_df.columns
global_mean = all_df['positive_score'].mean()


df_koelectra = pd.read_csv('../data/KoELECTRA_X_feature_v2.2_ratio.csv', encoding='utf-8-sig')

keywords = ["family", "anniversary", "date", "distance", "revisit",
             "service", "ambiance", "view", "taste", 
             "price", "waiting", "luxury", "disappoint"]

score_cols = [c for c in df_koelectra.columns if c.endswith("_score")]
count_cols = [c for c in df_koelectra.columns if c.endswith("_count")]
m = 20

bayes_cols_to_keep = ['매장명'] # 병합을 위해 키(Key) 변수인 매장명 보관
for score_col in score_cols:
    base = score_col[:-6]  # "_score" 제거
    count_col = base + "_count"
    bayes_col = base + "_bayes"

    if count_col in df_koelectra.columns:
        df_koelectra[bayes_col] = (
            (df_koelectra[count_col] * df_koelectra[score_col] + m * global_mean)
            / (df_koelectra[count_col] + m)
        )
        bayes_cols_to_keep.append(bayes_col)

print(f"✅ 계산된 Bayes 변수 개수: {len(bayes_cols_to_keep) - 1}개")


In [ ]:
print("\n[ 🚀 LightGBM + Optuna를 활용한 블루리본 예측 모델링 ]")

# =========================================================================
# 1. 데이터 불러오기 및 X, y 세팅
# =========================================================================
df = pd.read_csv('../data/블루리본_최최종_마참내_v1.1.1.csv', encoding='cp949')
# 💡 계산된 베이지안 컬럼(df_koelectra)을 매장명 기준으로 df에 병합
df = pd.merge(df, df_koelectra[bayes_cols_to_keep], on='매장명', how='left')
print(f"ℹ️ 베이지안 변수 병합 후 컬럼 수: {len(df.columns)}")
y = df['블루리본 여부']

# 이미지 모델 결과 컬럼들의 0 값을 NaN으로 변경 (LightGBM 특화 처리)
image_cols = ['고급성_평균', '쾌적성_평균', '감성_평균']
df[image_cols] = df[image_cols].replace(0, np.nan)
print("ℹ️ 이미지 변수 내 0 -> NaN 변환 완료!")

# 'has_missing' 이라는 새로운 파생 변수 생성
# 결측값이 하나라도 있으면 1, 완전한 데이터면 0으로 채워집니다.
df['has_missing'] = df.isnull().any(axis=1).astype(int)

# 💡 결과가 잘 들어갔는지 확인해보기
print(df['has_missing'].value_counts())

# '카테고리' 컬럼을 원-핫 인코딩합니다.
# dtype=int를 넣어야 True/False가 아닌 1/0 형태로 깔끔하게 들어갑니다.
df = pd.get_dummies(df, columns=['카테고리'], prefix='카테고리', dtype=int)

In [ ]:
# --- 변수 제거 설정 ---
# 기본적으로 제거할 변수
base_drop_cols = ['매장명', '블루리본 여부', 'idx_family_weighted_score',
                 'total_review_count', '통과된_내부사진_개수', '총_사진_개수']
base_score_cols = [col for col in df.columns if col.endswith('_score')]
base_ratio_cols = [col for col in df.columns if col.endswith('_ratio')]
base_max_cols = [col for col in df.columns if col.endswith('_최대')]
base_mid_cols = [col for col in df.columns if col.endswith('_중앙값')]

base_drop_cols = base_drop_cols + base_score_cols + base_ratio_cols + base_max_cols + base_mid_cols
final_drop_cols = list(set(base_drop_cols))
X_base = df.drop(columns=final_drop_cols)

print("\n✅ 다음 변수들을 제거하고 모델링을 시작합니다:")
for col in sorted(final_drop_cols):
    if col in df.columns:
        print(f" - {col}")
print(f"✅ 사용된 총 변수 개수: {len(X_base.columns)}개")

In [ ]:
# =========================================================================
# 2. Train / Test 분리
# =========================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42, stratify=y
)

- n_estimators : 150 ~ 400 -> 100 ~ 250
- learning_rate : 0.01 ~ 0.05 -> 0.01 ~ 0.05
- num_leaves (최대 리프 수): 10 ~ 20 ➔ 3 ~ 7
- max_depth (최대 깊이): 3 ~ 5 ➔ 2 ~ 3

- subsample: 0.65 ~ 0.85 ➔ 0.60 ~ 0.80
- subsample_freq: 1 ~ 3 ➔ 1 ~ 3
- colsample_bytree: 0.6 ~ 0.8 ➔ 0.50 ~ 0.80
  
- min_child_samples: 30 ~ 50 ➔ 50 ~ 80
- min_child_weight: 0.1 ~ 5.0 ➔ 0.1 ~ 5.0

- reg_alpha / reg_lambda (규제): 1.0 ~ 4.0 ➔ 2.0 ~ 10.0

- min_split_gain: 0.1 ~ 0.5 ➔ 0.1 ~ 0.5

- scoring: roc_auc ➔ f1

In [ ]:
# =========================================================================
# 3. Optuna를 사용한 하이퍼파라미터 튜닝
# =========================================================================
print("\n[ 🔎 Optuna를 사용하여 최적의 하이퍼파라미터 탐색 시작 ]")

N_TRIALS = 100      # 탐색 횟수 (시간이 오래 걸리면 50으로 줄여서 테스트)
CV_SPLITS = 5

cv = StratifiedKFold(
    n_splits=CV_SPLITS,
    shuffle=True,
    random_state=42
)

def objective_lgbm_regularized(trial):
    params = {
        # 기본 설정
        "objective": "binary",
        "metric": "binary_logloss",
        "boosting_type": "gbdt",
        "random_state": 42,
        "verbose": -1,
        "n_jobs": -1,

        # 1. 모델 복잡도 (논리적 정렬 및 상한선 축소)
        "n_estimators": trial.suggest_int("n_estimators", 150, 400), # 불필요한 반복 학습 차단
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.05, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 3), # 깊이를 보수적으로 제한
        "num_leaves": trial.suggest_int("num_leaves", 7, 15), # 💡 max_depth에 맞춰 최대 15로 제한

        # 2. 과적합 방지용 샘플링
        "subsample": trial.suggest_float("subsample", 0.65, 0.85),
        "subsample_freq": trial.suggest_int("subsample_freq", 1, 3),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.10, 0.30),

        # 3. 리프 최소 데이터 수 (하한선 상향)
        "min_child_samples": trial.suggest_int("min_child_samples", 50, 150), # 💡 조금 더 깐깐하게
        "min_child_weight": trial.suggest_float("min_child_weight", 0.1, 5.0, log=True),

        # 4. 규제 (현재 잘 작동하고 있으므로 범위만 최적화)
        "reg_alpha": trial.suggest_float("reg_alpha", 5.0, 50.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 5.0, 50.0),

        # 5. 분할 이득 허들
        "min_split_gain": trial.suggest_float("min_split_gain", 0.1, 0.5)
    }

    model = lgb.LGBMClassifier(**params)
    
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )
    
    return np.mean(scores)

study = optuna.create_study(
    direction="maximize",
    study_name="LGBM_Optimization"
)

study.optimize(
    objective_lgbm_regularized,
    n_trials=N_TRIALS,
    show_progress_bar=True
)

best_params = study.best_params

print("\n" + "=" * 90)
print("🏆 Optuna 튜닝 완료")
print(f"Best CV ROC-AUC: {study.best_value:.4f}")
print("\nBest Hyperparameters:")
print(best_params)
print("=" * 90)

# print("\n[ 📊 Optuna 시각화 그래프 생성 중... ]")

# # 1. 최적화 탐색 이력 (점수가 어떻게 올라갔는지 확인)
# fig_history = vis.plot_optimization_history(study)
# fig_history.show()

# # 2. 하이퍼파라미터 중요도 (어떤 변수가 성능에 가장 큰 영향을 줬는지 확인)
# fig_importance = vis.plot_param_importances(study)
# fig_importance.show()

# # 3. 하이퍼파라미터 평행 좌표계 (성능이 좋은 파라미터들의 조합 확인)
# fig_parallel = vis.plot_parallel_coordinate(study)
# fig_parallel.show()

# # 4. 개별 파라미터 변화에 따른 성능 산점도 (선택 사항)
# fig_slice = vis.plot_slice(study)
# fig_slice.show()

🎀 블루리본 매장 추천 (현재 상황): 사용자에게 "여긴 무조건 맛집!"이라고 추천해야 한다면, 기준을 0.5에서 0.6이나 0.65로 높이는 것이 합리적입니다. 51%의 애매한 확률을 가진 식당을 추천했다가 사용자가 실망하는 위험을 줄이기 위해서입니다

##### threshold 통합 버전

In [ ]:
# # 사용자가 직접 지정한 고정 파라미터 
# best_params = {
#     "n_estimators": 176,
#     "learning_rate": 0.0167,
#     "max_depth": 4,
#     "num_leaves": 19,
#     "subsample": 0.723,          # 72.3%
#     "subsample_freq": 1,
#     "colsample_bytree": 0.628,   # 62.8%
#     "min_child_samples": 35,
#     "min_child_weight": 0.4544,
#     "reg_alpha": 2.401,          # L1 규제
#     "reg_lambda": 2.557,         # L2 규제
#     "min_split_gain": 0.174
# }

In [ ]:
# =========================================================================
# 4. 최적 파라미터로 최종 모델 학습 
# =========================================================================
print("\n[ 🚀 최종 모델 학습 시작 ]")

final_model = lgb.LGBMClassifier(
    **best_params, # Optuna에서 찾은 최적 파라미터
    objective="binary",
    metric="binary_logloss",
    boosting_type="gbdt",
    random_state=42,
    verbose=-1,
    n_jobs=-1
)

final_model.fit(X_train, y_train)

# 확률값 계산 (AUC 계산 및 Threshold 조정에 공통으로 사용)
y_train_proba = final_model.predict_proba(X_train)[:, 1]
y_test_proba = final_model.predict_proba(X_test)[:, 1]

# =========================================================================
# 5. 모델 기본 체력 및 과적합 검증 (AUC 중심)
# =========================================================================
train_auc = roc_auc_score(y_train, y_train_proba)
test_auc = roc_auc_score(y_test, y_test_proba)
auc_gap = abs(train_auc - test_auc)

# 기본 임계값(0.5) 기준의 정확도
train_acc_default = accuracy_score(y_train, (y_train_proba >= 0.5).astype(int))
test_acc_default = accuracy_score(y_test, (y_test_proba >= 0.5).astype(int))

print("\n[ 🎯 Train vs Test 과적합 검증 (Default Threshold: 0.5) ]")
print(f"Train ROC-AUC : {train_auc:.4f}")
print(f"Test ROC-AUC  : {test_auc:.4f}")
print(f"AUC Gap       : {auc_gap:.4f} (0.05 이하 권장)")
print(f"Train Accuracy: {train_acc_default:.4f}")
print(f"Test Accuracy : {test_acc_default:.4f}")

# =========================================================================
# 6. 비즈니스 최적화를 위한 Threshold 탐색
# =========================================================================
print("\n[ 🔍 다양한 Threshold에 따른 실전 성능 변화 테스트 ]")
print("Threshold | Accuracy | Precision | Recall | F1-Score")
print("-" * 55)

for threshold in np.arange(0.50, 0.72, 0.02):
    temp_pred = (y_test_proba >= threshold).astype(int)
    
    acc = accuracy_score(y_test, temp_pred)
    prec = precision_score(y_test, temp_pred, zero_division=0)
    rec = recall_score(y_test, temp_pred)
    f1 = f1_score(y_test, temp_pred)
    
    print(f"   {threshold:.2f}   |  {acc:.4f}  |   {prec:.4f}  | {rec:.4f} |  {f1:.4f}")

In [ ]:
# =========================================================================
# 7. 최종 Threshold 적용 및 상세 리포트 출력
# =========================================================================
# 💡 위 루프 결과를 확인한 후, 가장 마음에 드는 숫자를 아래에 입력하세요!
FINAL_THRESHOLD = 0.56

# Train과 Test 각각에 새로운 Threshold 적용
final_train_pred = (y_train_proba >= FINAL_THRESHOLD).astype(int)
final_test_pred = (y_test_proba >= FINAL_THRESHOLD).astype(int)

print(f"\n[ 📊 최종 실전 성능 (적용된 Threshold: {FINAL_THRESHOLD}) ]")
# 💡 최종 Train 정확도 출력 추가
print(f"최종 Train Accuracy : {accuracy_score(y_train, final_train_pred):.4f}")
print(f"최종 Test Accuracy  : {accuracy_score(y_test, final_test_pred):.4f}")
# 최종 Accuracy Gap 확인 (선택 사항)
print(f"Accuracy Gap        : {abs(accuracy_score(y_train, final_train_pred) - accuracy_score(y_test, final_test_pred)):.4f}")
print("-" * 55)
print(classification_report(y_test, final_test_pred, zero_division=0))

In [ ]:
# =========================================================================
# 5. 특성 중요도 (Feature Importance) 확인
# =========================================================================
importance_gain = final_model.booster_.feature_importance(importance_type='gain')

importance_df = pd.DataFrame({
    'Feature': X_train.columns, 
    'Importance': importance_gain
}).sort_values('Importance', ascending=False).head(15)

# print("\n[ 🔥 상위 15개 중요 변수 ]")
# print(importance_df.to_string(index=False))

# 시각화
fig, ax = plt.subplots(figsize=(12, 8))
lgb.plot_importance(final_model, ax=ax, max_num_features=20, height=0.8, 
                    importance_type='gain', title='LightGBM Feature Importance (Gain)')
plt.tight_layout()
plt.show()

In [ ]:
importance_df

### 리뷰 적은 거 제거

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    classification_report, roc_auc_score, f1_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import optuna.visualization as vis

# 한글 폰트 설정
plt.rc('font', family='Malgun Gothic') 
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
print("\n[ 🚀 LightGBM + Optuna를 활용한 블루리본 예측 모델링 ]")

# =========================================================================
# 1. 데이터 불러오기 및 X, y 세팅
# =========================================================================
df = pd.read_csv('../data/블루리본_최최종_마참내_v1.1.csv', encoding='cp949')
# 리뷰 적은 음식점 제외
df = df.loc[df.loc[:, 'total_review_count'] >= 50]
y = df['블루리본 여부']

# 이미지 모델 결과 컬럼들의 0 값을 NaN으로 변경 (LightGBM 특화 처리)
image_cols = ['고급성_평균', '쾌적성_평균', '감성_평균']
df[image_cols] = df[image_cols].replace(0, np.nan)
print("ℹ️ 이미지 변수 내 0 -> NaN 변환 완료!")

# 'has_missing' 이라는 새로운 파생 변수 생성
# 결측값이 하나라도 있으면 1, 완전한 데이터면 0으로 채워집니다.
df['has_missing'] = df.isnull().any(axis=1).astype(int)

# 💡 결과가 잘 들어갔는지 확인해보기
print(df['has_missing'].value_counts())

# '카테고리' 컬럼을 원-핫 인코딩합니다.
# dtype=int를 넣어야 True/False가 아닌 1/0 형태로 깔끔하게 들어갑니다.
df = pd.get_dummies(df, columns=['카테고리'], prefix='카테고리', dtype=int)

In [ ]:
# --- 변수 제거 설정 ---
# 기본적으로 제거할 변수
# '_weighted_score'는 살리고, 순수하게 '_score'로 끝나는 컬럼만 추출
base_score_cols = [
    col for col in df.columns 
    if col.endswith('_score') and not col.endswith('_weighted_score')
]
count_cols = [col for col in df.columns if col.endswith('_count')]
ratio_cols = [col for col in df.columns if col.endswith('_ratio')]
base_drop_cols = [
    '매장명', '블루리본 여부', 'idx_family_weighted_score', 
    'total_review_count', '총_사진_개수', '통과된_내부사진_개수', 
    '고급성_중앙값', '쾌적성_중앙값', '감성_중앙값',
    '고급성_최대', '쾌적성_최대', '감성_최대'
]

final_drop_cols = list(set(base_score_cols + count_cols + ratio_cols + base_drop_cols))
X_base = df.drop(columns=final_drop_cols)

print("\n✅ 다음 변수들을 제거하고 모델링을 시작합니다:")
for col in sorted(final_drop_cols):
    if col in df.columns:
        print(f" - {col}")
print(f"✅ 사용된 총 변수 개수: {len(X_base.columns)}개")


In [ ]:
# =========================================================================
# 2. Train / Test 분리
# =========================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42, stratify=y
)

- n_estimators : 150 ~ 400 -> 100 ~ 250
- learning_rate : 0.01 ~ 0.05 -> 0.01 ~ 0.05
- num_leaves (최대 리프 수): 10 ~ 20 ➔ 3 ~ 7
- max_depth (최대 깊이): 3 ~ 5 ➔ 2 ~ 3

- subsample: 0.65 ~ 0.85 ➔ 0.60 ~ 0.80
- subsample_freq: 1 ~ 3 ➔ 1 ~ 3
- colsample_bytree: 0.6 ~ 0.8 ➔ 0.50 ~ 0.80
  
- min_child_samples: 30 ~ 50 ➔ 50 ~ 80
- min_child_weight: 0.1 ~ 5.0 ➔ 0.1 ~ 5.0

- reg_alpha / reg_lambda (규제): 1.0 ~ 4.0 ➔ 2.0 ~ 10.0

- min_split_gain: 0.1 ~ 0.5 ➔ 0.1 ~ 0.5

- scoring: roc_auc ➔ f1

In [ ]:
# =========================================================================
# 3. Optuna를 사용한 하이퍼파라미터 튜닝
# =========================================================================
print("\n[ 🔎 Optuna를 사용하여 최적의 하이퍼파라미터 탐색 시작 ]")

N_TRIALS = 100      # 탐색 횟수 (시간이 오래 걸리면 50으로 줄여서 테스트)
CV_SPLITS = 5

cv = StratifiedKFold(
    n_splits=CV_SPLITS,
    shuffle=True,
    random_state=42
)

def objective_lgbm_regularized(trial):
    params = {
        # 기본 설정
        "objective": "binary",
        "metric": "binary_logloss",
        "boosting_type": "gbdt",
        "random_state": 42,
        "verbose": -1,
        "n_jobs": -1,

        # 1. 모델 복잡도 (논리적 정렬 및 상한선 축소)
        "n_estimators": trial.suggest_int("n_estimators", 150, 400), # 불필요한 반복 학습 차단
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.05, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 5), # 깊이를 보수적으로 제한
        "num_leaves": trial.suggest_int("num_leaves", 10, 20), # 💡 max_depth에 맞춰 최대 15로 제한

        # 2. 과적합 방지용 샘플링
        "subsample": trial.suggest_float("subsample", 0.65, 0.85),
        "subsample_freq": trial.suggest_int("subsample_freq", 1, 3),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.60, 0.80),

        # 3. 리프 최소 데이터 수 (하한선 상향)
        "min_child_samples": trial.suggest_int("min_child_samples", 30, 50), # 💡 조금 더 깐깐하게
        "min_child_weight": trial.suggest_float("min_child_weight", 0.1, 5.0, log=True),

        # 4. 규제 (현재 잘 작동하고 있으므로 범위만 최적화)
        "reg_alpha": trial.suggest_float("reg_alpha", 1.0, 4.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 4.0),

        # 5. 분할 이득 허들
        "min_split_gain": trial.suggest_float("min_split_gain", 0.1, 0.5)
    }

    model = lgb.LGBMClassifier(**params)
    
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )
    
    return np.mean(scores)

study = optuna.create_study(
    direction="maximize",
    study_name="LGBM_Optimization"
)

study.optimize(
    objective_lgbm_regularized,
    n_trials=N_TRIALS,
    show_progress_bar=True
)

best_params = study.best_params

print("\n" + "=" * 90)
print("🏆 Optuna 튜닝 완료")
print(f"Best CV ROC-AUC: {study.best_value:.4f}")
print("\nBest Hyperparameters:")
print(best_params)
print("=" * 90)

print("\n[ 📊 Optuna 시각화 그래프 생성 중... ]")

# 1. 최적화 탐색 이력 (점수가 어떻게 올라갔는지 확인)
fig_history = vis.plot_optimization_history(study)
fig_history.show()

# 2. 하이퍼파라미터 중요도 (어떤 변수가 성능에 가장 큰 영향을 줬는지 확인)
fig_importance = vis.plot_param_importances(study)
fig_importance.show()

# 3. 하이퍼파라미터 평행 좌표계 (성능이 좋은 파라미터들의 조합 확인)
fig_parallel = vis.plot_parallel_coordinate(study)
fig_parallel.show()

# 4. 개별 파라미터 변화에 따른 성능 산점도 (선택 사항)
fig_slice = vis.plot_slice(study)
fig_slice.show()

🎀 블루리본 매장 추천 (현재 상황): 사용자에게 "여긴 무조건 맛집!"이라고 추천해야 한다면, 기준을 0.5에서 0.6이나 0.65로 높이는 것이 합리적입니다. 51%의 애매한 확률을 가진 식당을 추천했다가 사용자가 실망하는 위험을 줄이기 위해서입니다

##### threshold 통합 버전

In [ ]:
# =========================================================================
# 4. 최적 파라미터로 최종 모델 학습 
# =========================================================================
print("\n[ 🚀 최종 모델 학습 시작 ]")

final_model = lgb.LGBMClassifier(
    **best_params, # Optuna에서 찾은 최적 파라미터
    objective="binary",
    metric="binary_logloss",
    boosting_type="gbdt",
    random_state=42,
    verbose=-1,
    n_jobs=-1
)

final_model.fit(X_train, y_train)

# 확률값 계산 (AUC 계산 및 Threshold 조정에 공통으로 사용)
y_train_proba = final_model.predict_proba(X_train)[:, 1]
y_test_proba = final_model.predict_proba(X_test)[:, 1]

# =========================================================================
# 5. 모델 기본 체력 및 과적합 검증 (AUC 중심)
# =========================================================================
train_auc = roc_auc_score(y_train, y_train_proba)
test_auc = roc_auc_score(y_test, y_test_proba)
auc_gap = abs(train_auc - test_auc)

# 기본 임계값(0.5) 기준의 정확도
train_acc_default = accuracy_score(y_train, (y_train_proba >= 0.5).astype(int))
test_acc_default = accuracy_score(y_test, (y_test_proba >= 0.5).astype(int))

print("\n[ 🎯 Train vs Test 과적합 검증 (Default Threshold: 0.5) ]")
print(f"Train ROC-AUC : {train_auc:.4f}")
print(f"Test ROC-AUC  : {test_auc:.4f}")
print(f"AUC Gap       : {auc_gap:.4f} (0.05 이하 권장)")
print(f"Train Accuracy: {train_acc_default:.4f}")
print(f"Test Accuracy : {test_acc_default:.4f}")

# =========================================================================
# 6. 비즈니스 최적화를 위한 Threshold 탐색
# =========================================================================
print("\n[ 🔍 다양한 Threshold에 따른 실전 성능 변화 테스트 ]")
print("Threshold | Accuracy | Precision | Recall | F1-Score")
print("-" * 55)

for threshold in np.arange(0.50, 0.72, 0.02):
    temp_pred = (y_test_proba >= threshold).astype(int)
    
    acc = accuracy_score(y_test, temp_pred)
    prec = precision_score(y_test, temp_pred, zero_division=0)
    rec = recall_score(y_test, temp_pred)
    f1 = f1_score(y_test, temp_pred)
    
    print(f"   {threshold:.2f}   |  {acc:.4f}  |   {prec:.4f}  | {rec:.4f} |  {f1:.4f}")

In [ ]:
# =========================================================================
# 7. 최종 Threshold 적용 및 상세 리포트 출력
# =========================================================================
# 💡 위 루프 결과를 확인한 후, 가장 마음에 드는 숫자를 아래에 입력하세요!
FINAL_THRESHOLD = 0.56

# Train과 Test 각각에 새로운 Threshold 적용
final_train_pred = (y_train_proba >= FINAL_THRESHOLD).astype(int)
final_test_pred = (y_test_proba >= FINAL_THRESHOLD).astype(int)

print(f"\n[ 📊 최종 실전 성능 (적용된 Threshold: {FINAL_THRESHOLD}) ]")
# 💡 최종 Train 정확도 출력 추가
print(f"최종 Train Accuracy : {accuracy_score(y_train, final_train_pred):.4f}")
print(f"최종 Test Accuracy  : {accuracy_score(y_test, final_test_pred):.4f}")
# 최종 Accuracy Gap 확인 (선택 사항)
print(f"Accuracy Gap        : {abs(accuracy_score(y_train, final_train_pred) - accuracy_score(y_test, final_test_pred)):.4f}")
print("-" * 55)
print(classification_report(y_test, final_test_pred, zero_division=0))

In [ ]:
# =========================================================================
# 5. 특성 중요도 (Feature Importance) 확인
# =========================================================================
importance_gain = final_model.booster_.feature_importance(importance_type='gain')

importance_df = pd.DataFrame({
    'Feature': X_train.columns, 
    'Importance': importance_gain
}).sort_values('Importance', ascending=False).head(15)

# print("\n[ 🔥 상위 15개 중요 변수 ]")
# print(importance_df.to_string(index=False))

# 시각화
fig, ax = plt.subplots(figsize=(12, 8))
lgb.plot_importance(final_model, ax=ax, max_num_features=20, height=0.8, 
                    importance_type='gain', title='LightGBM Feature Importance (Gain)')
plt.tight_layout()
plt.show()

In [ ]:
importance_df